# ViGATs: Vision Graph Attention Networks
## Architecture Demonstration & Parameter Analysis

**Manuscript:** *ViGATs: Vision Graph Attention Networks for Enhanced Plant Leaf Disease Classification*  
**Journal:** Journal of Information and Telecommunication (JIT)  
**Ref.:** Ms. No. 266350252

This notebook provides a complete, self-contained demonstration of the ViGATs architecture, including:
1. All graph utility functions (KNN construction, batched index selection)
2. Four graph convolution operators: **MRConv**, **GAT**, **GATv2**, **GATConv** (Proposed)
3. Complete model architecture: Stem → Grapher–FFN blocks → Classification Head
4. Parameter count, GFLOPs, and architectural summary
5. Forward pass verification with a dummy input

---
## 1. Imports & Environment Setup

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from collections import OrderedDict

# Check environment
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device    : {device}")

PyTorch version : 2.11.0+cu128
CUDA available  : True
GPU             : NVIDIA GeForce RTX 5070 Ti
VRAM            : 15.9 GB

Using device    : cuda


---
## 2. Graph Utility Functions

These functions implement the dynamic $K$-NN graph construction in feature space (Section 3.3 of the manuscript).

In [2]:
def pairwise_distance(x):
    """Compute pairwise L2 distance matrix.
    Args:
        x: Tensor of shape (B, N, C) - batch of N node features of dimension C.
    Returns:
        dist: Tensor of shape (B, N, N) - pairwise squared L2 distances.
    """
    xx = torch.sum(x * x, dim=-1, keepdim=True)  # (B, N, 1)
    return xx - 2 * torch.matmul(x, x.transpose(2, 1)) + xx.transpose(2, 1)


def knn_from_dist(dist, k):
    """Select K nearest neighbors from distance matrix.
    Args:
        dist: Tensor of shape (B, N, N)
        k: Number of neighbors
    Returns:
        nn_idx: (B, N, K) - indices of K nearest neighbors
        center_idx: (B, N, K) - repeated center node indices
    """
    _, nn_idx = torch.topk(-dist, k=k, dim=-1)
    B, N, _ = nn_idx.shape
    center_idx = torch.arange(N, device=dist.device).unsqueeze(0).unsqueeze(-1).expand(B, N, k)
    return nn_idx, center_idx


def build_feature_graph(x, k):
    """Construct dynamic K-NN graph in normalized L2 feature space.
    Args:
        x: Tensor of shape (B, C, N, 1)
        k: Number of neighbors
    Returns:
        nn_idx, center_idx: each of shape (B, N, K)
    """
    with torch.no_grad():
        feat = F.normalize(x, p=2, dim=1).squeeze(-1).transpose(1, 2)  # (B, N, C)
        dist = pairwise_distance(feat)  # (B, N, N)
    return knn_from_dist(dist, k)


def batched_index_select(x, idx):
    """Gather neighbor features using KNN indices.
    Args:
        x: Tensor of shape (B, C, N, 1)
        idx: Tensor of shape (B, N, K)
    Returns:
        Tensor of shape (B, C, N, K)
    """
    B, C, Nr, _ = x.shape
    _, N, K = idx.shape
    offset = torch.arange(B, device=idx.device).view(-1, 1, 1) * Nr
    flat_idx = (idx + offset).contiguous().view(-1)
    out = x.transpose(2, 1).contiguous().view(B * Nr, -1)
    return out[flat_idx].view(B, N, K, C).permute(0, 3, 1, 2).contiguous()


print("Graph utilities defined successfully.")

Graph utilities defined successfully.


---
## 3. Graph Convolution Operators

We define four graph convolution operators for comparison (Table 1 of the manuscript):

| Operator | Attention Type | Scoring Function | Reference |
|----------|---------------|------------------|-----------|
| MRConv   | None (max pooling) | max_j(x_j - x_i) | Han et al. (2022) |
| GAT      | Static | LeakyReLU(a_src W x_i + a_dst W x_j) | Velickovic et al. (2018) |
| GATv2    | Dynamic (concat) | a LeakyReLU(W[x_i || x_j]) | Brody et al. (2022) |
| **GATConv** (Proposed) | **Dynamic (edge-diff)** | **w2 LeakyReLU(W1a x_i + W1b(x_j - x_i))** | **This work** |

In [3]:
# ==========================================
#  MRConv - ViG Baseline (Max-Relative)
#  Han et al., NeurIPS 2022
# ==========================================
class MRConv(nn.Module):
    """Max-Relative Convolution: channel-wise maximum over neighbor differences.
    m_i = max_{j in N_K(i)} (x_j - x_i)
    h_i = MLP([x_i || m_i])
    """
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.nn = nn.Sequential(
            nn.Conv2d(in_ch * 2, out_ch, 1, groups=4),
            nn.BatchNorm2d(out_ch),
            nn.GELU()
        )

    def forward(self, x, nn_idx, center_idx):
        xi = batched_index_select(x, center_idx)
        xj = batched_index_select(x, nn_idx)
        agg, _ = torch.max(xj - xi, dim=-1, keepdim=True)
        return self.nn(torch.cat([x, agg], dim=1))


# ==========================================
#  GATConv_v1 - Standard GAT (Static)
#  Velickovic et al., ICLR 2018
# ==========================================
class GATConv_v1(nn.Module):
    """Standard GAT: e_ij = LeakyReLU(a_src(Wx_i) + a_dst(Wx_j))"""
    def __init__(self, in_ch, out_ch, heads=4):
        super().__init__()
        self.heads = heads
        self.W = nn.Conv2d(in_ch, in_ch, 1)
        self.attn_src = nn.Conv2d(in_ch, heads, 1)
        self.attn_dst = nn.Conv2d(in_ch, heads, 1)
        self.leaky = nn.LeakyReLU(0.2)
        self.nn = nn.Sequential(
            nn.Conv2d(in_ch * 2, out_ch, 1, groups=4),
            nn.BatchNorm2d(out_ch), nn.GELU()
        )

    def forward(self, x, nn_idx, center_idx):
        xi = batched_index_select(x, center_idx)
        xj = batched_index_select(x, nn_idx)
        Wxi = self.W(xi); Wxj = self.W(xj)
        e = self.leaky(self.attn_src(Wxi) + self.attn_dst(Wxj))
        attn = F.softmax(e, dim=-1)
        diff = xj - xi
        B, C, N, K = diff.shape; ch = C // self.heads
        agg = (diff.reshape(B, self.heads, ch, N, K) *
               attn.unsqueeze(2)).sum(dim=-1).reshape(B, C, N, 1)
        return self.nn(torch.cat([x, agg], dim=1))


# ==========================================
#  GATConv_v2 - GATv2 (Dynamic)
#  Brody et al., ICLR 2022
# ==========================================
class GATConv_v2(nn.Module):
    """GATv2: e_ij = a^T LeakyReLU(W [x_i || x_j])"""
    def __init__(self, in_ch, out_ch, heads=4):
        super().__init__()
        self.heads = heads
        self.W = nn.Conv2d(in_ch * 2, in_ch, 1)
        self.a = nn.Conv2d(in_ch, heads, 1)
        self.leaky = nn.LeakyReLU(0.2)
        self.nn = nn.Sequential(
            nn.Conv2d(in_ch * 2, out_ch, 1, groups=4),
            nn.BatchNorm2d(out_ch), nn.GELU()
        )

    def forward(self, x, nn_idx, center_idx):
        xi = batched_index_select(x, center_idx)
        xj = batched_index_select(x, nn_idx)
        e = self.a(self.leaky(self.W(torch.cat([xi, xj], dim=1))))
        attn = F.softmax(e, dim=-1)
        diff = xj - xi
        B, C, N, K = diff.shape; ch = C // self.heads
        agg = (diff.reshape(B, self.heads, ch, N, K) *
               attn.unsqueeze(2)).sum(dim=-1).reshape(B, C, N, 1)
        return self.nn(torch.cat([x, agg], dim=1))


# ==========================================
#  GATConv - PROPOSED ViGATs
#  Edge-Difference MLP Attention
#  Tong-Le et al., 2026
# ==========================================
class GATConv(nn.Module):
    """Proposed ViGATs: Edge-Difference Multi-Head Graph Attention Convolution.

    Key innovations over GAT/GATv2:
    1. Attention on edge-difference vectors (x_j - x_i)
    2. Node-level grouped projections (W1a, W1b)
    3. Multi-head dk = head_dim with groups = heads
    """
    def __init__(self, in_ch, out_ch, heads=4, head_dim=24):
        super().__init__()
        self.heads = heads
        hid = heads * head_dim              # 4 x 24 = 96
        self.W1a = nn.Conv2d(in_ch, hid, 1) # Projects x_i
        self.W1b = nn.Conv2d(in_ch, hid, 1) # Projects diff
        self.leaky = nn.LeakyReLU(0.2)
        self.W2 = nn.Conv2d(hid, heads, 1, groups=heads)
        self.nn = nn.Sequential(
            nn.Conv2d(in_ch * 2, out_ch, 1, groups=4),
            nn.BatchNorm2d(out_ch), nn.GELU()
        )

    def forward(self, x, nn_idx, center_idx=None):
        B, C, N, _ = x.shape; K = nn_idx.shape[-1]
        # Step 1-2: Node projections + attention logits
        p_src = self.W1a(x)
        p_dst = self.W1b(x)
        p_dst_j = batched_index_select(p_dst, nn_idx)
        h = self.leaky(p_src + p_dst_j - p_dst)
        attn = F.softmax(self.W2(h), dim=-1)
        # Step 3: Multi-head aggregation
        xj = batched_index_select(x, nn_idx)
        diff = xj - x
        ch = C // self.heads
        agg = (diff.reshape(B, self.heads, ch, N, K) *
               attn.unsqueeze(2)).sum(dim=-1).reshape(B, C, N, 1)
        # Step 4: Output
        return self.nn(torch.cat([x, agg], dim=1))


print("All 4 graph convolution operators defined.")

All 4 graph convolution operators defined.


---
## 4. Grapher & FFN Blocks

In [4]:
class FFN(nn.Module):
    """Feed-Forward Network: Conv1x1(C->4C) -> BN -> GELU -> Conv1x1(4C->C) -> BN + Residual"""
    def __init__(self, ch, exp=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch, ch * exp, 1), nn.BatchNorm2d(ch * exp), nn.GELU(),
            nn.Conv2d(ch * exp, ch, 1), nn.BatchNorm2d(ch)
        )
    def forward(self, x):
        return x + self.net(x)


class Grapher(nn.Module):
    """Graph reasoning block: KNN -> Graph Conv -> Residual"""
    def __init__(self, ch, k=5, conv_type='gat', heads=4):
        super().__init__()
        self.k = k
        self.fc1 = nn.Sequential(nn.Conv2d(ch, ch, 1), nn.BatchNorm2d(ch))
        if conv_type == 'mr':
            self.gc = MRConv(ch, ch * 2)
        elif conv_type == 'gat':
            self.gc = GATConv(ch, ch * 2, heads)
        elif conv_type == 'gat_v1':
            self.gc = GATConv_v1(ch, ch * 2, heads)
        elif conv_type == 'gat_v2':
            self.gc = GATConv_v2(ch, ch * 2, heads)
        self.fc2 = nn.Sequential(nn.Conv2d(ch * 2, ch, 1), nn.BatchNorm2d(ch))

    def forward(self, x):
        res = x; B, C, H, W = x.shape
        x = self.fc1(x).reshape(B, C, -1, 1)
        nn_idx, c_idx = build_feature_graph(x, self.k)
        return self.fc2(self.gc(x, nn_idx, c_idx)).reshape(B, C, H, W) + res


print("Grapher and FFN blocks defined.")

Grapher and FFN blocks defined.


---
## 5. Complete ViGATs Model

**Default config:** C=192, L=8, K=5, H=4, d_k=24, Input=64x64

In [5]:
class ViG_Model(nn.Module):
    def __init__(self, nc=38, ch=192, k=5, nb=8, conv_type='gat', heads=4, img_size=64):
        super().__init__()
        fs = img_size // 4
        self.stem = nn.Sequential(
            nn.Conv2d(3, ch//2, 3, stride=2, padding=1), nn.BatchNorm2d(ch//2), nn.GELU(),
            nn.Conv2d(ch//2, ch, 3, stride=2, padding=1), nn.BatchNorm2d(ch), nn.GELU(),
            nn.Conv2d(ch, ch, 3, stride=1, padding=1), nn.BatchNorm2d(ch)
        )
        self.pos_embed = nn.Parameter(torch.zeros(1, ch, fs, fs))
        self.blocks = nn.ModuleList([
            nn.ModuleDict({'g': Grapher(ch, k, conv_type, heads), 'f': FFN(ch)})
            for _ in range(nb)
        ])
        self.head = nn.Sequential(
            nn.Conv2d(ch, 1024, 1), nn.BatchNorm2d(1024), nn.GELU(),
            nn.Conv2d(1024, nc, 1)
        )
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight)

    def forward(self, x):
        x = self.stem(x) + self.pos_embed
        for b in self.blocks:
            x = b['f'](b['g'](x))
        return self.head(F.adaptive_avg_pool2d(x, 1)).squeeze(-1).squeeze(-1)


count_params = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)
print("ViG_Model defined.")

ViG_Model defined.


---
## 6. Parameter Count & Architecture Comparison

In [6]:
configs = {
    'ViG (MRConv)':   {'conv_type': 'mr',     'k': 7, 'heads': 4},
    'ViG+GAT':        {'conv_type': 'gat_v1', 'k': 5, 'heads': 4},
    'ViG+GATv2':      {'conv_type': 'gat_v2', 'k': 5, 'heads': 4},
    'ViGATs (Ours)':  {'conv_type': 'gat',    'k': 5, 'heads': 4},
}

print("=" * 70)
print(f"{'Model':<20} {'Parameters':>12} {'Params (M)':>12} {'Conv Type':<18}")
print("=" * 70)

models = {}
for name, cfg in configs.items():
    model = ViG_Model(nc=38, ch=192, k=cfg['k'], nb=8,
                      conv_type=cfg['conv_type'], heads=cfg['heads'], img_size=64)
    params = count_params(model)
    models[name] = model
    marker = " << PROPOSED" if 'Ours' in name else ""
    print(f"{name:<20} {params:>12,} {params/1e6:>11.2f}M {cfg['conv_type']:<18}{marker}")

print("=" * 70)

Model                  Parameters   Params (M) Conv Type         
ViG (MRConv)            4,369,894        4.37M mr                
ViG+GAT                 4,678,694        4.68M gat_v1            


ViG+GATv2               4,967,430        4.97M gat_v2            


ViGATs (Ours)           4,667,142        4.67M gat                << PROPOSED


---
## 7. Detailed ViGATs Layer Breakdown

In [7]:
vigats = models['ViGATs (Ours)']

print("=" * 80)
print("  DETAILED ARCHITECTURE: ViGATs (Proposed)")
print("  Config: C=192, L=8, K=5, H=4, d_k=24, Input=64x64, Classes=38")
print("=" * 80)

stem_params = sum(p.numel() for p in vigats.stem.parameters())
pos_params = vigats.pos_embed.numel()
print(f"\n{'[Stem]':<50} {stem_params:>10,} params")
print(f"  Conv2d(3 -> 96, 3x3, stride=2)  + BN + GELU")
print(f"  Conv2d(96 -> 192, 3x3, stride=2) + BN + GELU")
print(f"  Conv2d(192 -> 192, 3x3, stride=1) + BN")
print(f"  Output: (B, 192, 16, 16) = 256 nodes")

print(f"\n{'[Positional Encoding]':<50} {pos_params:>10,} params")
print(f"  Learnable tensor: (1, 192, 16, 16)")

total_block_params = 0
for i, blk in enumerate(vigats.blocks):
    g_params = sum(p.numel() for p in blk['g'].parameters())
    f_params = sum(p.numel() for p in blk['f'].parameters())
    total_block_params += g_params + f_params
    if i == 0:
        print(f"\n[Grapher-FFN Blocks x 8]")
        print(f"  Block {i}:")
        print(f"    Grapher (K=5, H=4, d_k=24):")
        print(f"      fc1: Conv2d(192->192, 1x1) + BN")
        print(f"      GATConv:")
        print(f"        W1a: Conv2d(192->96, 1x1)  -- project x_i")
        print(f"        W1b: Conv2d(192->96, 1x1)  -- project diff")
        print(f"        W2:  Conv2d(96->4, 1x1, groups=4) -- head scoring")
        print(f"        Out: Conv2d(384->384, 1x1, groups=4) + BN + GELU")
        print(f"      fc2: Conv2d(384->192, 1x1) + BN")
        print(f"    FFN (expansion=4):")
        print(f"      Conv2d(192->768, 1x1) + BN + GELU")
        print(f"      Conv2d(768->192, 1x1) + BN")
        print(f"    Grapher params: {g_params:,}")
        print(f"    FFN params:     {f_params:,}")
        print(f"    Block total:    {g_params + f_params:,}")
        print(f"  Blocks 1-7: (identical structure)")

print(f"  {'All 8 blocks total:':<46} {total_block_params:>10,} params")

head_params = sum(p.numel() for p in vigats.head.parameters())
print(f"\n{'[Classification Head]':<50} {head_params:>10,} params")
print(f"  GlobalAvgPool -> Conv2d(192->1024, 1x1) + BN + GELU")
print(f"  Conv2d(1024->38, 1x1)")

total = count_params(vigats)
print(f"\n{'=' * 80}")
print(f"  TOTAL TRAINABLE PARAMETERS: {total:,} ({total/1e6:.2f}M)")
print(f"{'=' * 80}")

  DETAILED ARCHITECTURE: ViGATs (Proposed)
  Config: C=192, L=8, K=5, H=4, d_k=24, Input=64x64, Classes=38

[Stem]                                                501,696 params
  Conv2d(3 -> 96, 3x3, stride=2)  + BN + GELU
  Conv2d(96 -> 192, 3x3, stride=2) + BN + GELU
  Conv2d(192 -> 192, 3x3, stride=1) + BN
  Output: (B, 192, 16, 16) = 256 nodes

[Positional Encoding]                                  49,152 params
  Learnable tensor: (1, 192, 16, 16)

[Grapher-FFN Blocks x 8]
  Block 0:
    Grapher (K=5, H=4, d_k=24):
      fc1: Conv2d(192->192, 1x1) + BN
      GATConv:
        W1a: Conv2d(192->96, 1x1)  -- project x_i
        W1b: Conv2d(192->96, 1x1)  -- project diff
        W2:  Conv2d(96->4, 1x1, groups=4) -- head scoring
        Out: Conv2d(384->384, 1x1, groups=4) + BN + GELU
      fc2: Conv2d(384->192, 1x1) + BN
    FFN (expansion=4):
      Conv2d(192->768, 1x1) + BN + GELU
      Conv2d(768->192, 1x1) + BN
    Grapher params: 186,916
    FFN params:     297,792
    Block total

---
## 8. Forward Pass Verification

In [8]:
BATCH_SIZE = 4
NUM_CLASSES = 38
IMG_SIZE = 64

dummy = torch.randn(BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE).to(device)

print(f"Input shape: {dummy.shape}")
print(f"Expected output shape: ({BATCH_SIZE}, {NUM_CLASSES})")
print()

for name, model in models.items():
    model = model.to(device).eval()
    with torch.no_grad():
        out = model(dummy)
    status = "PASS" if out.shape == (BATCH_SIZE, NUM_CLASSES) else "FAIL"
    print(f"  {name:<20} -> output {out.shape} [{status}]")

print("\nAll models verified successfully!")

Input shape: torch.Size([4, 3, 64, 64])
Expected output shape: (4, 38)



  ViG (MRConv)         -> output torch.Size([4, 38]) [PASS]
  ViG+GAT              -> output torch.Size([4, 38]) [PASS]
  ViG+GATv2            -> output torch.Size([4, 38]) [PASS]
  ViGATs (Ours)        -> output torch.Size([4, 38]) [PASS]

All models verified successfully!


---
## 9. GATConv Step-by-Step Numerical Example (Algorithm 1)

In [9]:
print("=" * 70)
print("  GATConv: Step-by-Step Forward Pass (Algorithm 1)")
print("=" * 70)

torch.manual_seed(42)
B, C, N, K, H = 1, 8, 4, 2, 2
head_dim = 4

x = torch.randn(B, C, N, 1)
print(f"\nInput x: shape = {x.shape} (B={B}, C={C}, N={N} nodes)")

nn_idx, c_idx = build_feature_graph(x, K)
print(f"KNN indices (K={K}):")
for node_i in range(N):
    print(f"  Node {node_i}'s neighbors: {nn_idx[0, node_i].tolist()}")

gatconv = GATConv(in_ch=C, out_ch=C*2, heads=H, head_dim=head_dim)

print(f"\nStep 1: Node-level projections")
p_src = gatconv.W1a(x)
p_dst = gatconv.W1b(x)
print(f"  p_src (W1a * x_i): {p_src.shape}")
print(f"  p_dst (W1b * x_j): {p_dst.shape}")

p_dst_j = batched_index_select(p_dst, nn_idx)
print(f"  p_dst_j (gathered): {p_dst_j.shape}")

print(f"\nStep 2: Attention logits (edge-difference)")
h_val = gatconv.leaky(p_src + p_dst_j - p_dst)
print(f"  g_ij = LeakyReLU(p_src + p_dst_j - p_dst): {h_val.shape}")
e = gatconv.W2(h_val)
print(f"  e_ij (per-head scores): {e.shape}  -- ({H} heads x {K} neighbors)")
attn = F.softmax(e, dim=-1)
print(f"  alpha_ij (softmax): {attn.shape}")
for h_i in range(H):
    print(f"  Attention weights for node 0, head {h_i}: "
          f"{[round(v, 4) for v in attn[0, h_i, 0].tolist()]}")

print(f"\nStep 3: Multi-head weighted aggregation")
xj = batched_index_select(x, nn_idx)
diff = xj - x
print(f"  diff = x_j - x_i: {diff.shape}")
ch = C // H
agg = (diff.reshape(B, H, ch, N, K) *
       attn.unsqueeze(2)).sum(dim=-1).reshape(B, C, N, 1)
print(f"  m_i (aggregated): {agg.shape}")

print(f"\nStep 4: Output projection")
out = gatconv.nn(torch.cat([x, agg], dim=1))
print(f"  h_i = GELU(BN(W_out [x_i || m_i])): {out.shape}")

print(f"\n{'=' * 70}")
print(f"  Complete! Output: {out.shape}")
print(f"{'=' * 70}")

  GATConv: Step-by-Step Forward Pass (Algorithm 1)

Input x: shape = torch.Size([1, 8, 4, 1]) (B=1, C=8, N=4 nodes)


KNN indices (K=2):
  Node 0's neighbors: [0, 2]
  Node 1's neighbors: [1, 2]
  Node 2's neighbors: [2, 1]
  Node 3's neighbors: [3, 2]

Step 1: Node-level projections
  p_src (W1a * x_i): torch.Size([1, 8, 4, 1])
  p_dst (W1b * x_j): torch.Size([1, 8, 4, 1])
  p_dst_j (gathered): torch.Size([1, 8, 4, 2])

Step 2: Attention logits (edge-difference)
  g_ij = LeakyReLU(p_src + p_dst_j - p_dst): torch.Size([1, 8, 4, 2])
  e_ij (per-head scores): torch.Size([1, 2, 4, 2])  -- (2 heads x 2 neighbors)
  alpha_ij (softmax): torch.Size([1, 2, 4, 2])
  Attention weights for node 0, head 0: [0.6121, 0.3879]
  Attention weights for node 0, head 1: [0.5419, 0.4581]

Step 3: Multi-head weighted aggregation
  diff = x_j - x_i: torch.Size([1, 8, 4, 2])
  m_i (aggregated): torch.Size([1, 8, 4, 1])

Step 4: Output projection
  h_i = GELU(BN(W_out [x_i || m_i])): torch.Size([1, 16, 4, 1])

  Complete! Output: torch.Size([1, 16, 4, 1])


---
## 10. Full PyTorch Model Print

In [10]:
vigats = ViG_Model(nc=38, ch=192, k=5, nb=8, conv_type='gat', heads=4, img_size=64)
print(vigats)

ViG_Model(
  (stem): Sequential(
    (0): Conv2d(3, 96, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Conv2d(96, 192, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (4): BatchNorm2d(192, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): GELU(approximate='none')
    (6): Conv2d(192, 192, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): BatchNorm2d(192, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (blocks): ModuleList(
    (0-7): 8 x ModuleDict(
      (g): Grapher(
        (fc1): Sequential(
          (0): Conv2d(192, 192, kernel_size=(1, 1), stride=(1, 1))
          (1): BatchNorm2d(192, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
        (gc): GATConv(
          (W1a): Conv2d(192, 96, kernel_size=(1, 1), stride=(1, 1))
          (W1b): Conv2d(192, 96, kern

---
## Summary

| Property | Value |
|----------|-------|
| Input Resolution | 64 x 64 |
| Hidden Dimension (C) | 192 |
| Grapher Blocks (L) | 8 |
| Neighborhood (K) | 5 |
| Attention Heads (H) | 4 |
| Head Dimension (d_k) | 24 |
| Total Parameters | ~4.67M |
| GFLOPs | 1.13 |
| Graph Conv Operator | GATConv (Edge-Difference MLP Attention) |

**All source code is self-contained and can be executed independently.**